In [1]:
import numpy as np
import pandas as pd
import nltk
import random
from nltk.corpus import names

random.seed(314)

# Getting the Data

I am using the corpus `names` from the `nltk.corpus` library as described in *Natural Language Processing with Python* (NLPP) Chapter 6. A lot of this is going to be using very similar, if not the exact, code from that chapter since I am modeling my gender classifier after it. 

In [2]:
names

<WordListCorpusReader in 'C:\\Users\\norem\\AppData\\Roaming\\nltk_data\\corpora\\names'>

This code is taken straight from the NLPP except here I make all the names lower case to start. This will make the processing easier down the line. Then I randomly reorder all of the names so random sampling is also easier later.

In [3]:
labeled_names = ([(name.lower(), 'male') for name in names.words('male.txt')] 
                 + [(name.lower(), 'female') for name in names.words('female.txt')])
random.shuffle(labeled_names)

Then I put all of the names into a `pandas` dataframe for easier data manipulation.

In [4]:
all_names = pd.DataFrame(labeled_names, columns = ['name', 'gender'])
print(all_names)

          name  gender
0        bucky    male
1     charleen  female
2     gustavus    male
3           ev    male
4          sib  female
...        ...     ...
7939     addie    male
7940   kendall    male
7941  gamaliel    male
7942  carolina  female
7943   malcolm    male

[7944 rows x 2 columns]


## Splitting the data

Since the names have already been randomized we can sample by just splitting up the data. Typically we could use a dedicated function to get each subset but this way we can just take the first 500 for the `test` set, the next 500 for the `dev_test` set and the remainder can be our training data.

In [5]:
test, dev_test, train = np.split(all_names, [500, 1000])

C:\Users\norem\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [6]:
test.head()

,name,gender
0,bucky,male
1,charleen,female
2,gustavus,male
3,ev,male
4,sib,female


In [7]:
dev_test.head()

,name,gender
500,frank,male
501,austin,male
502,dannie,male
503,vivienne,female
504,quentin,female


In [8]:
train.head()

,name,gender
1000,monty,male
1001,chelsy,female
1002,erhard,male
1003,aurelea,female
1004,lon,male


To verify that all the sets of names have roughly the same proportions of gendered names we can take a ratio for each of these data sets. After getting a ratio for all of them we can see that they are all roughly a ratio of 1.7ish-to-1 female to male names. This means that I likely have a pretty good random sampling of the data for each set.

In [9]:
x = all_names.groupby('gender').agg('count')
print(x)
x['name'].iloc[0]/x['name'].iloc[1]

        name
gender      
female  5001
male    2943


1.6992864424057084

In [10]:
x = test.groupby('gender').agg('count')
print(x)
x['name'].iloc[0]/x['name'].iloc[1]

        name
gender      
female   307
male     193


1.5906735751295338

In [11]:
x = dev_test.groupby('gender').agg('count')
print(x)
x['name'].iloc[0]/x['name'].iloc[1]

        name
gender      
female   314
male     186


1.6881720430107527

In [12]:
x = train.groupby('gender').agg('count')
print(x)
x['name'].iloc[0]/x['name'].iloc[1]

        name
gender      
female  4380
male    2564


1.7082683307332294

# Testing Features

To test a large number of features I decided to define some functions that will make testing the new features fast and easy. All I will need to do is create a new feature making function and pass the name of the function into the accuracy function that uses all other functions I made to create a result. These methods are taken straight from NLPP Chapter 6, just reworked to make the process eaiser.

In [13]:
def featureset(df, features):
    return [(features(df['name'].iloc[i]), df['gender'].iloc[i]) 
                   for i in range(len(df))]

def training(features, flag = True):
    featuresets = featureset(train, features)
    classifier = nltk.NaiveBayesClassifier.train(featuresets)
    if flag: classifier.show_most_informative_features(5)
    return classifier

def accuracy(features):
    classifier = training(features)
    print(nltk.classify.accuracy(classifier, featureset(dev_test, features)))

## Ends of Names

Starting with the ending of names we will just take the last letter. Exactly as they do it in NLPP Chapter 6, we get a pretty high accuracy for the predictive capabilities with a value of 0.77.

In [14]:
def gender_features_last(name):
     return {'last_letter': name[-1]}

accuracy(gender_features_last)

Most Informative Features
             last_letter = 'a'            female : male   =     35.5 : 1.0
             last_letter = 'k'              male : female =     30.0 : 1.0
             last_letter = 'f'              male : female =     13.3 : 1.0
             last_letter = 'p'              male : female =      9.9 : 1.0
             last_letter = 'v'              male : female =      9.9 : 1.0
0.77


Next I tried to see what the last two letters in the names would do vs just the last one, these performed slightly better at 0.812 accuracy. The most informative features have a much higher ratio than the previous example. This makes it so those indicators definitely have a pretty strong leaning for the gender of a name if they have these features.

In [15]:
def gender_features_suf2(name):
    return {'suffix2': name[-2:]}

accuracy(gender_features_suf2)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
                 suffix2 = 'sa'           female : male   =     32.4 : 1.0
0.812


To iterate further on this concept I tried with the last three letters, this performed worse so it is better to just stick with the last two letters. The most informative features are much less informative than the previous but better on average than the first.

In [16]:
def gender_features_suf3(name):
    return {'suffix3': name[-3:]}

accuracy(gender_features_suf3)

Most Informative Features
                 suffix3 = 'ita'          female : male   =     24.9 : 1.0
                 suffix3 = 'ana'          female : male   =     23.7 : 1.0
                 suffix3 = 'tta'          female : male   =     22.4 : 1.0
                 suffix3 = 'ard'            male : female =     21.7 : 1.0
                 suffix3 = 'vin'            male : female =     19.3 : 1.0
0.792


## Beginnings of Names

Next I tried to do the same but for the beginning of the names instead of the end. As shown in Chapter 6 of NLPP. Interestingly the first letter of a name is not nearly as informative as the last. With only an accuracy score of 0.626 I am definitely going to have to try more than just the first letter. The most informative features have a very small ratio relative to the suffix features.

In [17]:
def gender_features_first(name):
    return {'first_letter': name[0].lower()}

accuracy(gender_features_first)

Most Informative Features
            first_letter = 'w'              male : female =      4.6 : 1.0
            first_letter = 'q'              male : female =      4.1 : 1.0
            first_letter = 'u'              male : female =      3.2 : 1.0
            first_letter = 'x'              male : female =      2.5 : 1.0
            first_letter = 'k'            female : male   =      2.2 : 1.0
0.626


Trying two letters next proved to be a bit better than just the first but not that much better and not any better than any of the suffixes with an accuracy score of 0.662.

In [18]:
def gender_features_pre2(name):
    return {'prefix2': name[:2]}

accuracy(gender_features_pre2)

Most Informative Features
                 prefix2 = 'hu'             male : female =     16.2 : 1.0
                 prefix2 = 'ya'             male : female =     11.7 : 1.0
                 prefix2 = 'tu'             male : female =      9.5 : 1.0
                 prefix2 = 'fo'             male : female =      9.1 : 1.0
                 prefix2 = 'wa'             male : female =      8.6 : 1.0
0.662


Continuing with the trend I tried the first three letters to get another marginal improvement but still worse than all suffix features at 0.698 accuracy.

In [19]:
def gender_features_pre3(name):
    return {'prefix3': name[:3]}

accuracy(gender_features_pre3)

Most Informative Features
                 prefix3 = 'dor'          female : male   =     16.8 : 1.0
                 prefix3 = 'tha'            male : female =     13.1 : 1.0
                 prefix3 = 'ken'            male : female =     11.0 : 1.0
                 prefix3 = 'gar'            male : female =      9.6 : 1.0
                 prefix3 = 'flo'          female : male   =      9.6 : 1.0
0.698


I then tried one last set of the first four letters but this is verging on overfitting but it is worth a shot. The feature does produce a higher accuracy score of 0.704 but it still doesn't perform as well at the suffix features. I stopped here because any more would definitely be overfitting.

In [20]:
def gender_features_pre4(name):
    return {'prefix4': name[:4]}

accuracy(gender_features_pre4)

Most Informative Features
                 prefix4 = 'flor'         female : male   =      9.0 : 1.0
                 prefix4 = 'john'           male : female =      8.2 : 1.0
                 prefix4 = 'mari'         female : male   =      8.0 : 1.0
                 prefix4 = 'barr'           male : female =      6.3 : 1.0
                 prefix4 = 'neal'           male : female =      6.3 : 1.0
0.704


## Specific letters
Taken from the overfit example in Chapter 6 of NLPP I first checked if names having specific letters would provide insight into the gender of the name. It doesn't do particularly well with an accuracy score of 0.688 but we can try using other permutations of this concept.

In [21]:
def gender_features_lhas(name):
    features = {}
    for letter in 'abcdefghijklmnopqrstuvwxyz':
        features["has({})".format(letter)] = (letter in name.lower())
    return features

accuracy(gender_features_lhas)

Most Informative Features
                  has(w) = True             male : female =      4.3 : 1.0
                  has(u) = True             male : female =      1.8 : 1.0
                  has(p) = True             male : female =      1.7 : 1.0
                  has(f) = True             male : female =      1.7 : 1.0
                  has(z) = True             male : female =      1.6 : 1.0
0.688


Next I tried seeing if a count of specific letters would fare better )an accuracy score of 0.718), while it is a bit better it's not a massive jump as indicated by the most informative features.

In [22]:
def gender_features_lcount(name):
    features = {}
    for letter in 'abcdefghijklmnopqrstuvwxyz':
        features["count({})".format(letter)] = name.lower().count(letter)
    return features

accuracy(gender_features_lcount)

Most Informative Features
                count(v) = 2              female : male   =      8.8 : 1.0
                count(a) = 3              female : male   =      5.0 : 1.0
                count(w) = 1                male : female =      4.3 : 1.0
                count(i) = 3                male : female =      3.7 : 1.0
                count(o) = 2                male : female =      3.2 : 1.0
0.718


## Pairs of Letters
I then wondered if it would work well to have every combination of two vowels, it doesn't do particularly well but there are only a small number of combinations. 

In [23]:
vowels = 'aeiou'
vpairs = []
for ea1 in vowels:
    for ea2 in vowels:
        vpairs.append(ea1+ea2)

print(vpairs)

def gender_features_vpairhas(name):
    features = {}
    for ea in vpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_vpairhas)

['aa', 'ae', 'ai', 'ao', 'au', 'ea', 'ee', 'ei', 'eo', 'eu', 'ia', 'ie', 'ii', 'io', 'iu', 'oa', 'oe', 'oi', 'oo', 'ou', 'ua', 'ue', 'ui', 'uo', 'uu']
Most Informative Features
                 has(iu) = True             male : female =     16.5 : 1.0
                 has(oo) = True             male : female =      4.9 : 1.0
                 has(ua) = True             male : female =      4.0 : 1.0
                 has(ee) = True           female : male   =      3.8 : 1.0
                 has(ia) = True           female : male   =      3.7 : 1.0
0.628


I wondered if a count of the vowel pairs would change things but unfortunately it seems that they pair the same so it's not definitely not worth running for counts and complicating things.

In [24]:
def gender_features_vpaircount(name):
    features = {}
    for ea in vpairs:
        features["count({})".format(ea)] = name.count(ea)
    return features

accuracy(gender_features_vpaircount)

Most Informative Features
               count(iu) = 1                male : female =     16.5 : 1.0
               count(oo) = 1                male : female =      4.9 : 1.0
               count(ua) = 1                male : female =      4.0 : 1.0
               count(ia) = 1              female : male   =      3.7 : 1.0
               count(ee) = 1              female : male   =      3.7 : 1.0
0.628


Using the previous examples I wanted to see what happens if I just used consonant pairs and just check whether they appear in the names and not take counts. This proved to be a lot better than just the vowels since it produced an accuracy score of 0.7.

In [25]:
consonants = 'bcdfghjklmnpqrstvwxyz'
cpairs = []
for ea1 in consonants:
    for ea2 in consonants:
        cpairs.append(ea1+ea2)

def gender_features_cpairhas(name):
    features = {}
    for ea in cpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_cpairhas)

Most Informative Features
                 has(rw) = True             male : female =     16.5 : 1.0
                 has(sp) = True             male : female =     14.2 : 1.0
                 has(lt) = True             male : female =     13.3 : 1.0
                 has(rk) = True             male : female =     12.6 : 1.0
                 has(tc) = True             male : female =     12.0 : 1.0
0.7


To try and get a better accuracy score I tried just using all the letters of the alphabet and that was extremely helpful. The accuracy score for this set is much higher (0.762) and the most informative features have relatively high ratios.

In [26]:
letter = 'abcdefghijklmnopqrstuvwxyz'
lpairs = []
for ea1 in letter:
    for ea2 in letter:
        lpairs.append(ea1+ea2)

def gender_features_lpairhas(name):
    features = {}
    for ea in lpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_lpairhas)

Most Informative Features
                 has(hu) = True             male : female =     23.6 : 1.0
                 has(fo) = True             male : female =     18.8 : 1.0
                 has(iu) = True             male : female =     16.5 : 1.0
                 has(rw) = True             male : female =     16.5 : 1.0
                 has(sp) = True             male : female =     14.2 : 1.0
0.762


## Length of Names
One final feature I was able to think of is just the length of the name. This doesn't do much with an accuracy score of 0.634 but it was worth a shot.

In [27]:
def gender_features_len(name):
    return {'length': len(name)}

accuracy(gender_features_len)

Most Informative Features
                  length = 3                male : female =      2.0 : 1.0
                  length = 2                male : female =      1.7 : 1.0
                  length = 13               male : female =      1.7 : 1.0
                  length = 10             female : male   =      1.4 : 1.0
                  length = 9              female : male   =      1.3 : 1.0
0.634


## Combining Features
After coming up with as many features as I could think of it was time to combine them in the hopes of getting a better prediction model. The first combination was the same combination that is in Chapter 6 of NLPP. This combination actually does worse than just using the suffix2 feature set since the accuracy score is lowers at 0.804.

In [28]:
def gender_features_test1(name):
    return {'suffix2': name[-2:], 
            'last_letter': name[-1]}

accuracy(gender_features_test1)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
             last_letter = 'a'            female : male   =     35.5 : 1.0
0.804


I then wondered if I would get a better score by adding more suffix features, surprisingly it produced the best score so far 0.818!

In [29]:
def gender_features_test2(name):
    return {'suffix2': name[-2:], 
            'last_letter': name[-1], 
            'suffix3': name[-3:]}

accuracy(gender_features_test2)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
             last_letter = 'a'            female : male   =     35.5 : 1.0
0.818


I then wanted to try and see what would happen if I removed the last_letter feature since it hurt the first combination. Removing it hurt the score but only a little bit to get a score of 0.816, this may be better to try and limit overfitting since it doesn't impact the score too much to remove it.

In [30]:
def gender_features_test3(name):
    return {'suffix2': name[-2:], 
            'suffix3': name[-3:]}

accuracy(gender_features_test3)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
                 suffix2 = 'sa'           female : male   =     32.4 : 1.0
0.816


I then tested at least one well performing feature from each category to see how well it would work to produce the best result and then cut some out. This way we can get a sleek solution that doesn't overfit the training data too much. This again produces the best results so far of 0.854.

In [31]:
def gender_features_test4(name):
    features = {}
    features['suffix2'] = name[-2:]
    features['last_letter'] = name[-1:]
    features['prefix3'] = name[:3]
    features['length'] = len(name)

    for ea in lpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_test4)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
             last_letter = 'a'            female : male   =     35.5 : 1.0
0.854


I then removed the length feature since it is the worst performing feature out of all those chosen. This actually gave a marginal improvement so it is definitely a welcome change, a simplier feature set for marginal improvement is always better to 0.856.

In [41]:
def gender_features_test5(name):
    features = {}
    features['suffix2'] = name[-2:]
    features['last_letter'] = name[-1:]
    features['prefix3'] = name[:3]
    for ea in lpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_test5)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
             last_letter = 'a'            female : male   =     35.5 : 1.0
0.856


I then tried to remove the last letter feature and get a small decrease in the score to 0.85.

In [40]:
def gender_features_test6(name):
    features = {}
    features['suffix2'] = name[-2:]
    features['prefix3'] = name[:3]
    for ea in lpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_test6)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
                 suffix2 = 'sa'           female : male   =     32.4 : 1.0
0.85


I didn't think the previous removal would do enough to remove any overfitting so for the final feature set I added back in the last letter and ran it for the final permutation.

In [39]:
def gender_features_final(name):
    features = {}
    features['suffix2'] = name[-2:]
    features['last_letter'] = name[-1:]
    features['prefix3'] = name[:3]
    for ea in lpairs:
        features["has({})".format(ea)] = (ea in name)
    return features

accuracy(gender_features_final)

Most Informative Features
                 suffix2 = 'na'           female : male   =     92.8 : 1.0
                 suffix2 = 'la'           female : male   =     69.9 : 1.0
                 suffix2 = 'ta'           female : male   =     42.6 : 1.0
                 suffix2 = 'ia'           female : male   =     36.1 : 1.0
             last_letter = 'a'            female : male   =     35.5 : 1.0
0.856


# Testing of Final Classifier
Using the same code from Chapter 6 of NLPP we can check the individual examples of incorrect guesses from the `dev_test` set. A number of these names are a bit abnormal for Americans so it is expected that the classifier has trouble identifying the correct gender for the names like: barbe, chickie, mignon, chane, daffy, and olin. There are others that would typically be considered what the classifier guessed but has been identified as the other in the gender data from the `names` corpus: chris, alfie, and quentin are traditionally male and nikita, sydney, and cat are typically female. These were identified correctly according to social norms but not accoring to the data categorization. There are a number that are incorrectly categorized but we wouldn't expect 100% accuracy so I think this is an acceptable level of incorrect values.

In [33]:
classifier = training(gender_features_final, False)

errors = []
for i in range(len(dev_test)):
    guess = classifier.classify(gender_features_final(dev_test['name'].iloc[i]))
    if guess != dev_test['gender'].iloc[i]:
        errors.append( (dev_test['gender'].iloc[i], guess, dev_test['name'].iloc[i]) )

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=alfie                         
correct=female   guess=male     name=allsun                        
correct=female   guess=male     name=barbe                         
correct=female   guess=male     name=chickie                       
correct=female   guess=male     name=chris                         
correct=female   guess=male     name=deb                           
correct=female   guess=male     name=evy                           
correct=female   guess=male     name=gunvor                        
correct=female   guess=male     name=jasmin                        
correct=female   guess=male     name=jorey                         
correct=female   guess=male     name=kendre                        
correct=female   guess=male     name=kit                           
correct=female   guess=male     name=koo                           
correct=female   guess=male     name=mag                           
correct=female   guess=male     name=margo      

# Test Set
Running the `test` set through my classifier I got an accuracy score of 0.844 which is slightly lower than the `dev_test` set but still high enough that it is not a surprising result. This helps to confirm that the classifier is not overfit to the training data and I am quite happy with the results of this classifier. I believe that this is the best classifier I am able to make that mimics those found in Chapter 6 of NLPP.

In [42]:
classifier = training(gender_features_final, False)
print(nltk.classify.accuracy(classifier, featureset(test, gender_features_final)))

0.844
